# RAM Graph Energy — sesión del benchmark principal

Ejecuta un shard de E2/E4 con presupuesto de tiempo. Requiere E0/E1 aprobados y el piloto medido. Para E4, E2 debe estar completo porque las continuaciones 25→100 reutilizan el checkpoint padre.


## 1. Instalar el paquete


In [ ]:
import os, pathlib, shutil, subprocess, sys

REPO_URL = "https://github.com/abelalv/energia-2.git"
REPO_DIR = pathlib.Path("/kaggle/working/energia-2")

if not (REPO_DIR / "pyproject.toml").exists():
    candidates = [
        path
        for path in pathlib.Path("/kaggle/input").rglob("pyproject.toml")
        if (path.parent / "src" / "ram_experiments").exists()
    ]
    if candidates:
        shutil.copytree(candidates[0].parent, REPO_DIR, dirs_exist_ok=True)
        print("repo copiado desde", candidates[0].parent)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
# Los subprocesos del notebook deben encontrar el paquete también sin instalación editable.
os.environ["PYTHONPATH"] = os.pathsep.join(
    [str(REPO_DIR / "src"), os.environ.get("PYTHONPATH", "")]
)
try:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"],
        check=True,
    )
except subprocess.CalledProcessError:
    # Kaggle sin red o sin uv_build: usar src/ directamente. El entorno sigue siendo el de Kaggle.
    sys.path.insert(0, str(REPO_DIR / "src"))
    print("pip install -e falló; usando src/ en sys.path")
print("cwd:", pathlib.Path.cwd())


## 2. Entorno


In [ ]:
import platform, torch

print("python:", platform.python_version())
print("torch:", torch.__version__)
print("cuda disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))
    print("vram GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

# Protocol RAM-2026-10-13-v1: float32 without AMP/TF32.
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
print("tf32:", torch.backends.cuda.matmul.allow_tf32)


## 3. Datos


In [ ]:
import os, pathlib, subprocess, sys

DATA_DIR = pathlib.Path(os.environ.get("RAM_DATA_DIR", "/kaggle/working/data/processed"))

if not (DATA_DIR / "mnist_main.npz").exists():
    attached = sorted(pathlib.Path("/kaggle/input").rglob("mnist_main.npz"))
    if attached:
        DATA_DIR = attached[0].parent
        print("datos adjuntos:", DATA_DIR)
    else:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "ram_experiments.cli",
                "prepare-data",
                "--datasets",
                "mnist,fashion_mnist",
                "--raw-dir",
                "/kaggle/working/data/raw",
                "--processed-dir",
                str(DATA_DIR),
            ],
            check=True,
        )

print("data dir:", DATA_DIR)
print(sorted(path.name for path in DATA_DIR.glob("*.npz")))
print("define RAM_DATA_DIR en las celdas de ejecución:", DATA_DIR)


## 4. Configuración de la sesión

Variables: `RAM_STAGE` (`E2` o `E4`) y `RAM_MAX_SECONDS`. La ejecución en 2 GPUs reparte los runs automáticamente; no toques `SHARD` ni `TOTAL_SHARDS` si usas esa opción.


In [ ]:
import os, pathlib, subprocess, sys

STAGE = os.environ.get("RAM_STAGE", "E2")          # E2 o E4
SHARD = int(os.environ.get("RAM_SHARD", "0"))      # solo para 1 GPU
TOTAL_SHARDS = int(os.environ.get("RAM_TOTAL_SHARDS", "1"))  # solo para 1 GPU
MAX_SECONDS = int(os.environ.get("RAM_MAX_SECONDS", "30600"))  # 8 h 30 min

MANIFEST = pathlib.Path("configs/main_manifest.jsonl")
RUN_ROOT = "/kaggle/working/runs"

if not MANIFEST.exists():
    subprocess.run(
        [
            sys.executable,
            "-m",
            "ram_experiments.cli",
            "manifest",
            "--stage",
            "main",
            "--out",
            str(MANIFEST),
        ],
        check=True,
    )
subprocess.run(
    [sys.executable, "-m", "ram_experiments.cli", "inventory", "--manifest", str(MANIFEST)],
    check=True,
)
print("STAGE:", STAGE, "MAX_SECONDS:", MAX_SECONDS)


## 5. Ejecución en 2 GPUs (recomendado en T4 x2)

Lanza **dos procesos independientes**, uno por GPU, con shards disjuntos. No es `DataParallel`: cada run conserva batch 64, semillas y perfil numérico. Primero precalcula las referencias STD y después reparte el resto del manifiesto.


In [ ]:
import subprocess, sys

# Dos procesos independientes, uno por GPU. No usa DataParallel: cada run conserva
# batch 64, semillas y perfil numérico del protocolo.
subprocess.run(
    [
        sys.executable,
        "scripts/run_dual_gpu.py",
        "--manifest",
        str(MANIFEST),
        "--stage",
        STAGE,
        "--run-root",
        RUN_ROOT,
        "--data-dir",
        str(DATA_DIR),
        "--max-seconds",
        str(MAX_SECONDS),
        "--gpus",
        "0,1",
    ],
    check=True,
)


## 6. Ejecución en 1 GPU (alternativa)

Úsala solo si no tienes dos GPUs o quieres repartir sesiones manualmente con `SHARD` y `TOTAL_SHARDS`.


In [ ]:
import subprocess, sys

# Alternativa de una sola GPU. SHARD y TOTAL_SHARDS permiten repartir sesiones.
subprocess.run(
    [
        sys.executable,
        "-m",
        "ram_experiments.cli",
        "run",
        "--manifest",
        str(MANIFEST),
        "--stage",
        STAGE,
        "--run-root",
        RUN_ROOT,
        "--data-dir",
        str(DATA_DIR),
        "--device",
        "cuda:0",
        "--shard",
        str(SHARD),
        "--total-shards",
        str(TOTAL_SHARDS),
        "--max-seconds",
        str(MAX_SECONDS),
        "--resume",
    ],
    check=True,
)


## 7. Agregación


In [ ]:
import subprocess, sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "ram_experiments.cli",
        "analyze",
        "--manifest",
        "configs/main_manifest.jsonl",
        "--run-root",
        "/kaggle/working/runs",
        "--out",
        "/kaggle/working/analysis/main",
    ],
    check=True,
)


## 8. Reanudación entre sesiones

Sube la carpeta `runs/` como dataset privado y móntala en `/kaggle/input/...`; copia solo los run directories incompletos a `/kaggle/working/runs` y vuelve a ejecutar la celda de 2 GPUs. El script usa `--resume`.
